# Modelo simple para etanol hidratado MT\n\nNotebook reproducible para integrar CEPEA, ANP, Brent y USD/BRL; comparar si Brent y USD/BRL agregan valor; y generar un forecast de 52 semanas con escenarios.

## Configuracion\n\nSi tienes una serie historica CEPEA completa, coloca un CSV con columnas `date` y `cepea_ethanol_mt_m3` y completa `CEPEA_CSV`. Sin ese archivo, el notebook usa la tabla publica visible de CEPEA, que puede no tener suficientes observaciones para modelar ese objetivo.

In [ ]:
from pathlib import Path\nimport sys\n\nimport pandas as pd\n\nPROJECT_ROOT = Path.cwd()\nsys.path.append(str(PROJECT_ROOT))\n\nfrom src.ethanol_mt_model import (\n    TARGETS, HORIZONS, load_dataset, backtest_direct, correlations,\n    forecast_recursive, variable_utility\n)\n\nSTART_YEAR = 2021\nCACHE_DIR = PROJECT_ROOT / 'data' / 'cache'\nOUTPUT_DIR = PROJECT_ROOT / 'outputs'\nCEPEA_CSV = ''  # ejemplo: 'data/cepea_mt_historico.csv'\n\nOUTPUT_DIR.mkdir(exist_ok=True)\nCACHE_DIR.mkdir(parents=True, exist_ok=True)

In [ ]:
dataset = load_dataset(\n    cache_dir=CACHE_DIR,\n    start_year=START_YEAR,\n    local_cepea_csv=Path(CEPEA_CSV) if CEPEA_CSV else None,\n)\ndataset.to_csv(OUTPUT_DIR / 'integrated_dataset.csv', index=False)\n\nprint(dataset.tail())\ndataset.notna().sum()

## Backtesting y utilidad real de variables\n\nSe comparan cuatro modelos: A solo rezagos/estacionalidad, B agrega gasolina MT, C agrega Brent, D agrega USD/BRL. La tabla `variable_utility.csv` muestra si cada bloque reduce MAE/RMSE frente al modelo anterior.

In [ ]:
backtests = []\ncorrs = []\n\nfor target in TARGETS:\n    usable = int(dataset[target].notna().sum())\n    print(f'{target}: {usable} observaciones no nulas')\n    if usable < 110:\n        print(f'  Saltado: no hay historia suficiente para rezagos de 52 semanas y backtest estable.')\n        continue\n    backtests.append(backtest_direct(dataset, target, horizons=HORIZONS))\n    corrs.append(correlations(dataset, target, max_lag=12))\n\nbacktest_df = pd.concat(backtests, ignore_index=True) if backtests else pd.DataFrame()\ncorr_df = pd.concat(corrs, ignore_index=True) if corrs else pd.DataFrame()\nutility_df = variable_utility(backtest_df)\n\nbacktest_df.to_csv(OUTPUT_DIR / 'backtest_metrics.csv', index=False)\ncorr_df.to_csv(OUTPUT_DIR / 'correlations.csv', index=False)\nutility_df.to_csv(OUTPUT_DIR / 'variable_utility.csv', index=False)\n\ndisplay(backtest_df.sort_values(['target', 'horizon_weeks', 'mae']).head(20))\ndisplay(utility_df)

## Forecast 52 semanas con escenarios\n\nEscenario base: Brent, USD/BRL y gasolina planos. Escenario `upside_macro`: Brent +15%, USD/BRL +8%, gasolina +5%. Escenario `downside_macro`: Brent -15%, USD/BRL -8%, gasolina -5%.

In [ ]:
forecast_frames = []\nscenarios = [\n    ('base', 0.00, 0.00, 0.00),\n    ('upside_macro', 0.15, 0.08, 0.05),\n    ('downside_macro', -0.15, -0.08, -0.05),\n]\n\nfor target in TARGETS:\n    if int(dataset[target].notna().sum()) < 110:\n        continue\n    for name, brent_shock, usd_shock, gasoline_shock in scenarios:\n        forecast_frames.append(\n            forecast_recursive(\n                dataset, target, name, weeks=52,\n                shock_brent=brent_shock,\n                shock_usd=usd_shock,\n                shock_gasoline=gasoline_shock,\n            )\n        )\n\nforecast_df = pd.concat(forecast_frames, ignore_index=True) if forecast_frames else pd.DataFrame()\nforecast_df.to_csv(OUTPUT_DIR / 'forecasts.csv', index=False)\ndisplay(forecast_df.head())\ndisplay(forecast_df.tail())

In [ ]:
try:\n    import matplotlib.pyplot as plt\n    for target in forecast_df['target'].dropna().unique():\n        ax = None\n        for scenario, group in forecast_df[forecast_df['target'].eq(target)].groupby('scenario'):\n            ax = group.plot(x='date', y='forecast', label=scenario, ax=ax, figsize=(10, 4))\n        ax.set_title(target)\n        ax.set_xlabel('date')\n        ax.set_ylabel('forecast')\n        plt.tight_layout()\n        plt.show()\nexcept ModuleNotFoundError:\n    print('matplotlib no esta instalado en este runtime; los CSV de salida ya fueron generados.')